# Teleport: chess positions -> Gazebo piece poses (library)

Pieces in the Gazebo world are static models moved only by the `set_pose` service ("vanishing
pieces"): a move = teleport the captured piece to the graveyard, then the moving piece to its
target square. This notebook keeps track of *which model instance* (`w_P_5`, `b_Q_2`, ...) stands
on which square, turns moves / FENs / whole PGN games into `set_pose` requests, and has backends
that send them (gz CLI, ROS 2 service).

**Deliverables** (README "Updated plan (11/4/2026)"): *PGN teleport replay script*, and
*Gazebo vanishing pieces to test game logic* (compare Gambit's moves to ground truth, verify
replayed games follow the rules: python-chess does the legality, we only move models).

**TODO checklist**
- [x] instance assignment for the start position and for any FEN (spare queens, graveyard)
- [x] move -> requests: captures, en passant, castling, promotion (spare queen / reuse captured)
- [x] PGN replay -> per-ply request lists
- [x] request formatting for `gz service` and `ros2 service call`
- [ ] `Ros2Backend.send` with rclpy (persistent node + client, much faster than the CLI)
- [ ] underpromotion when no captured piece of that type is available (add spare N/B/R models?)
- [ ] optional: animate (lift, move, drop) instead of an instant teleport, for nicer videos

In [ ]:
%run ./world_generator.ipynb

In [ ]:
import shutil
import subprocess
from dataclasses import dataclass, field

import chess
import chess.pgn

## A set_pose request

Gazebo's `/world/<world>/set_pose` takes a `gz.msgs.Pose` (name + position + orientation) and
returns `gz.msgs.Boolean`. Bridged to ROS 2 it is `ros_gz_interfaces/srv/SetEntityPose` with
`entity: {name, type: 2 (MODEL)}`. Orientation is a quaternion (x, y, z, w); identity = upright.

In [ ]:
@dataclass
class SetPose:
    name: str
    position: tuple                     # (x, y, z) meters, world frame
    orientation: tuple = (0.0, 0.0, 0.0, 1.0)   # quaternion (x, y, z, w)

    def gz_req(self):
        (x, y, z), (qx, qy, qz, qw) = self.position, self.orientation
        return (f'name: "{self.name}", position: {{x: {x:.4f}, y: {y:.4f}, z: {z:.4f}}}, '
                f'orientation: {{x: {qx:.6f}, y: {qy:.6f}, z: {qz:.6f}, w: {qw:.6f}}}')

    def gz_command(self, world=WORLD_NAME, timeout_ms=2000):
        return ["gz", "service", "-s", f"/world/{world}/set_pose", "--reqtype", "gz.msgs.Pose",
                "--reptype", "gz.msgs.Boolean", "--timeout", str(timeout_ms), "--req", self.gz_req()]

    def ros2_request(self):
        """Dict matching ros_gz_interfaces/srv/SetEntityPose.Request (entity type 2 = MODEL)."""
        (x, y, z), (qx, qy, qz, qw) = self.position, self.orientation
        return {"entity": {"name": self.name, "type": 2},
                "pose": {"position": {"x": x, "y": y, "z": z},
                         "orientation": {"x": qx, "y": qy, "z": qz, "w": qw}}}

    def ros2_command(self, world=WORLD_NAME):
        import json
        return ["ros2", "service", "call", f"/world/{world}/set_pose",
                "ros_gz_interfaces/srv/SetEntityPose", json.dumps(self.ros2_request())]

def color_char(color):
    return "w" if color == chess.WHITE else "b"

## Piece tracker

The tracker maps squares to instance names, and keeps the graveyard (slots along +x; the spare
queens start in slots 0..3, captured pieces take the lowest free slot).

Rules for promotions: first reuse a *captured* piece of the promoted type and color (sitting in
the graveyard), otherwise for a queen take an unused spare queen. Anything else is a TODO.

`apply_move(move)` returns the requests **and** pushes the move onto the internal board, so the
tracker always mirrors the game. Captured piece goes first so two models never overlap.

In [ ]:
@dataclass
class PieceTracker:
    board: chess.Board = field(default_factory=chess.Board)
    on_square: dict = field(default_factory=dict)     # chess square index -> instance name
    in_grave: dict = field(default_factory=dict)      # instance name -> graveyard slot

    @staticmethod
    def instance_kind(name):
        """'w_Q_2' -> (chess.WHITE, chess.QUEEN)."""
        c, k, _ = name.split("_")
        return (c == "w", chess.Piece.from_symbol(k).piece_type)

    @classmethod
    def from_fen(cls, fen=chess.STARTING_FEN, cfg=BOARD):
        """Assign model instances to the pieces of any position.

        Pieces on their start square get their start instance (so the start FEN reproduces the
        world exactly); the rest take free instances of the same color/kind in name order; unused
        instances go to the graveyard. Raises ValueError when there are not enough models
        (e.g. 3 knights of one color).
        """
        t = cls(board=chess.Board(fen))
        insts = start_instances(cfg)
        start_sq = {n: chess.parse_square(sq) for n, _, sq in insts if sq is not None}
        free = [n for n, _, _ in insts]
        pending = []
        for sq, piece in sorted(t.board.piece_map().items()):
            want = [n for n, s in start_sq.items() if s == sq and n in free
                    and t.instance_kind(n) == (piece.color, piece.piece_type)]
            if want:
                t.on_square[sq] = want[0]; free.remove(want[0])
            else:
                pending.append((sq, piece))
        for sq, piece in pending:
            cand = [n for n in free if t.instance_kind(n) == (piece.color, piece.piece_type)]
            if not cand:
                raise ValueError(f"no free model for {piece.symbol()} on {chess.square_name(sq)}")
            t.on_square[sq] = cand[0]; free.remove(cand[0])
        for slot, n in enumerate(free):
            t.in_grave[n] = slot
        return t

    # -- graveyard -------------------------------------------------------------------------
    def _free_slot(self):
        used = set(self.in_grave.values())
        return next(i for i in range(len(used) + 1) if i not in used)

    def _bury(self, name, cfg=BOARD):
        slot = self._free_slot()
        self.in_grave[name] = slot
        return SetPose(name, graveyard_pose(slot, cfg))

    def _take_for_promotion(self, color, piece_type):
        cands = sorted((s, n) for n, s in self.in_grave.items()
                       if self.instance_kind(n) == (color, piece_type))
        if not cands:
            # TODO: underpromotion with no captured piece of that type -> add spare N/B/R models
            # to the world (world_generator.start_instances) or fall back to a queen model.
            raise NotImplementedError(f"no model available to promote to {chess.piece_name(piece_type)}")
        # Prefer pieces that were captured (higher slots) over the original spares? Either is fine;
        # take the lowest slot so the spare queens are used first and the row stays tidy.
        name = cands[0][1]
        del self.in_grave[name]
        return name

    # -- moves -----------------------------------------------------------------------------
    def poses(self, cfg=BOARD):
        """Current {instance: (x, y, z)} for every instance (board + graveyard)."""
        out = {n: square_center(sq, cfg) for sq, n in self.on_square.items()}
        out.update({n: graveyard_pose(s, cfg) for n, s in self.in_grave.items()})
        return out

    def requests_for_all(self, cfg=BOARD):
        """Requests that put every instance where the tracker says (use after from_fen / reset)."""
        return [SetPose(n, p) for n, p in sorted(self.poses(cfg).items())]

    def apply_move(self, move, cfg=BOARD):
        """Push `move` (chess.Move or UCI/SAN string) and return the list of SetPose requests."""
        b = self.board
        if isinstance(move, str):
            try:
                move = chess.Move.from_uci(move)
            except ValueError:
                move = b.parse_san(move)
        if move not in b.legal_moves:
            raise ValueError(f"illegal move {move} in {b.fen()}")
        reqs = []
        frm, to = move.from_square, move.to_square
        mover = self.on_square.pop(frm)

        # 1. capture (en passant: the captured pawn is behind the target square)
        cap_sq = None
        if b.is_en_passant(move):
            cap_sq = to - 8 if b.turn == chess.WHITE else to + 8
        elif b.piece_at(to) is not None:
            cap_sq = to
        if cap_sq is not None:
            reqs.append(self._bury(self.on_square.pop(cap_sq), cfg))

        # 2. promotion: pawn leaves, another model arrives
        if move.promotion:
            reqs.append(self._bury(mover, cfg))
            mover = self._take_for_promotion(b.turn, move.promotion)

        # 3. the moving piece
        self.on_square[to] = mover
        reqs.append(SetPose(mover, square_center(to, cfg)))

        # 4. castling: the rook too
        if b.is_castling(move):
            rank = chess.square_rank(frm)
            rf, rt = (7, 5) if b.is_kingside_castling(move) else (0, 3)
            rook = self.on_square.pop(chess.square(rf, rank))
            self.on_square[chess.square(rt, rank)] = rook
            reqs.append(SetPose(rook, square_center(chess.square(rt, rank), cfg)))

        b.push(move)
        return reqs

    def check_consistent(self):
        """The tracker's squares/kinds must match python-chess's board exactly."""
        pm = self.board.piece_map()
        assert set(pm) == set(self.on_square), "occupied squares differ"
        for sq, p in pm.items():
            assert self.instance_kind(self.on_square[sq]) == (p.color, p.piece_type), chess.square_name(sq)
        assert not set(self.on_square.values()) & set(self.in_grave), "model on board and in grave"
        return True

## PGN replay

One entry per ply: index (1 = first move; 0 is the start position with all poses), SAN, UCI, FEN
after the move, requests. `run_replay_game.ipynb` sends these with a backend and (optionally)
grabs a camera frame after each ply.

In [ ]:
@dataclass
class Ply:
    idx: int
    san: str
    uci: str
    fen: str
    requests: list

def pgn_replay(pgn_text, cfg=BOARD):
    """PGN text (first game) -> [Ply]. Ply 0 resets every model to the game's start position."""
    import io
    game = chess.pgn.read_game(io.StringIO(pgn_text))
    if game is None:
        raise ValueError("no game in PGN text")
    t = PieceTracker.from_fen(game.board().fen(), cfg)
    plies = [Ply(0, "", "", t.board.fen(), t.requests_for_all(cfg))]
    for i, move in enumerate(game.mainline_moves(), start=1):
        san = t.board.san(move)
        reqs = t.apply_move(move, cfg)
        plies.append(Ply(i, san, move.uci(), t.board.fen(), reqs))
    return plies

def read_pgn_games(path):
    """Yield chess.pgn.Game objects from a (multi-game) PGN file."""
    with open(path) as f:
        while (g := chess.pgn.read_game(f)) is not None:
            yield g

## Backends

* `DryRunBackend` records requests (tests, notebooks without Gazebo).
* `GzCliBackend` shells out to `gz service` (simple, ~50 ms per call; fine for replays).
  Needs Gazebo on PATH; the world must be running.
* `Ros2Backend` uses the bridged ROS 2 service (start the bridge from docs/gazebo.md step 4,
  or `gambit.launch.py`, which starts it). TODO.

In [ ]:
class DryRunBackend:
    def __init__(self):
        self.sent = []
    def send(self, req):
        self.sent.append(req); return True
    def send_all(self, reqs):
        return all(self.send(r) for r in reqs)

class GzCliBackend(DryRunBackend):
    def __init__(self, world=WORLD_NAME, timeout_ms=2000):
        super().__init__()
        if shutil.which("gz") is None:
            raise RuntimeError("`gz` not found: source your ROS/Gazebo setup first")
        self.world, self.timeout_ms = world, timeout_ms
    def send(self, req):
        out = subprocess.run(req.gz_command(self.world, self.timeout_ms), capture_output=True, text=True)
        ok = out.returncode == 0 and "true" in out.stdout
        if not ok:
            raise RuntimeError(f"set_pose {req.name} failed: {out.stdout} {out.stderr}")
        self.sent.append(req); return True

class Ros2Backend(DryRunBackend):
    """TODO: rclpy client for /world/<world>/set_pose (ros_gz_interfaces/srv/SetEntityPose).

    Hints: try: import rclpy; from ros_gz_interfaces.srv import SetEntityPose; from
    ros_gz_interfaces.msg import Entity. Create one node + client in __init__, wait_for_service,
    then for each request fill req.entity.name / req.entity.type = Entity.MODEL / req.pose and
    rclpy.spin_until_future_complete(node, client.call_async(req), timeout_sec=2.0).
    See ros_playground.ipynb for a sketch.
    """
    def __init__(self, world=WORLD_NAME):
        super().__init__()
        try:
            import rclpy  # noqa: F401
        except ImportError as e:
            raise RuntimeError("rclpy not available: source /opt/ros/<distro>/setup.bash") from e
        raise NotImplementedError("TODO: Ros2Backend")

## Self-check

In [ ]:
def _close(a, b, tol=1e-4):
    return all(abs(x - y) < tol for x, y in zip(a, b))

# start FEN reproduces the world
t = PieceTracker.from_fen()
assert t.poses() == start_poses() and t.check_consistent()

# docs/gazebo.md: e2e4 -> w_P_5 to (0.2025, -0.0125, 0.006)
r = t.apply_move("e2e4")
assert len(r) == 1 and r[0].name == "w_P_5" and _close(r[0].position, (0.2025, -0.0125, 0.006))
assert 'name: "w_P_5"' in r[0].gz_req() and r[0].ros2_request()["entity"]["type"] == 2

# capture: 1.e4 d5 2.exd5 -> b_P_4 to graveyard slot 4 (slots 0..3 hold the spare queens)
t.apply_move("d7d5"); r = t.apply_move("exd5")
assert [q.name for q in r] == ["b_P_4", "w_P_5"] and _close(r[0].position, graveyard_pose(4))

# castling moves the rook as well
t2 = PieceTracker.from_fen("r3k2r/8/8/8/8/8/8/R3K2R w KQkq - 0 1")
r = t2.apply_move("e1g1")
assert [q.name for q in r] == ["w_K_1", "w_R_2"] and _close(r[1].position, square_center("f1"))

# en passant
t3 = PieceTracker.from_fen()
for m in ["e2e4", "a7a6", "e4e5", "d7d5"]:
    t3.apply_move(m)
r = t3.apply_move("e5d6")
assert r[0].name == "b_P_4" and _close(r[1].position, square_center("d6"))

# promotion uses a spare queen (first in graveyard)
t4 = PieceTracker.from_fen("8/P7/8/8/8/8/8/k6K w - - 0 1")
r = t4.apply_move("a7a8q")
assert r[0].name == "w_P_1" and r[1].name.startswith("w_Q_") and _close(r[1].position, square_center("a8"))
assert t4.check_consistent()

# PGN replay (Scholar's mate) keeps the tracker consistent with python-chess
plies = pgn_replay("1. e4 e5 2. Bc4 Nc6 3. Qh5 Nf6 4. Qxf7# 1-0")
assert len(plies) == 8 and len(plies[0].requests) == 36 and plies[-1].san == "Qxf7#"
assert [q.name for q in plies[-1].requests] == ["b_P_6", "w_Q_1"]
d = DryRunBackend(); assert d.send_all(plies[1].requests) and len(d.sent) == 1

# not enough models
try:
    PieceTracker.from_fen("NNN5/8/8/8/8/8/8/k6K w - - 0 1"); raise AssertionError
except ValueError:
    pass
print("teleport self-check OK")